# Session 5: Gaussian quadrature
**A. Real-world problem.** In Session 4 Romberg integration needed 65 function values to
reach 1e-12 for a bell curve, and Simpson's rule needed 513. If every value is a one-minute
simulation that is a long wait. All those rules used *equally spaced* points and only chose
the weights. What if we also choose **where** to sample? Today you build the Gauss-Legendre
rule: n well-chosen points that are exact for every polynomial up to degree 2n - 1, check
it against exact areas, compare it with Simpson and Romberg for the same number of values,
and find out when it fails.

*General education only: the numbers are for learning, not for engineering design.*

In [ ]:
import math
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: the points are the roots of a Legendre polynomial
The Legendre polynomials P_0 = 1, P_1 = x, P_2 = (3x^2 - 1)/2, ... come from the recurrence
(k + 1) P_(k+1) = (2k + 1) x P_k - k P_(k-1). The n sample points of the n-point Gauss rule
on [-1, 1] are the n roots of P_n; they lie inside the interval, crowd towards the ends and
are symmetric about 0. We find each root with Newton's method, starting from a good guess.

In [ ]:
def legendre_and_derivative(n, x):
    """P_n(x) and its derivative, by the three-term recurrence."""
    if n < 0:
        raise ValueError("n must be at least 0")
    p_prev, p = 1.0, x
    if n == 0:
        return 1.0, 0.0
    for k in range(1, n):
        p_prev, p = p, ((2 * k + 1) * x * p - k * p_prev) / (k + 1)
    dp = n * (x * p - p_prev) / (x * x - 1.0)
    return p, dp


def gauss_legendre_nodes(n):
    """Nodes (increasing) and weights of the n-point Gauss-Legendre rule on [-1, 1]."""
    if n < 1:
        raise ValueError("n must be at least 1")
    nodes, weights = [], []
    for k in range(1, n + 1):
        x = math.cos(math.pi * (k - 0.25) / (n + 0.5))      # good starting guess for the k-th root
        for _ in range(100):
            p, dp = legendre_and_derivative(n, x)
            step = p / dp
            x -= step
            if abs(step) < 1e-15:
                break
        _, dp = legendre_and_derivative(n, x)
        nodes.append(x)
        weights.append(2.0 / ((1.0 - x * x) * dp * dp))
    order = np.argsort(nodes)
    return np.array(nodes)[order], np.array(weights)[order]

## Step 2: the rule on any interval
Map [-1, 1] to [a, b] with x = m + r t (m = midpoint, r = half-length). The integral is
r times the weighted sum of f at the mapped points. A rule with n points costs exactly n
function values.

In [ ]:
def gauss_legendre(f, a, b, n):
    """Integral of f from a to b with the n-point Gauss-Legendre rule (n function values)."""
    if not a < b:
        raise ValueError("a must be smaller than b")
    t, w = gauss_legendre_nodes(n)
    m, r = 0.5 * (a + b), 0.5 * (b - a)
    return float(r * sum(wi * f(m + r * ti) for ti, wi in zip(t, w)))


def composite_gauss(f, edges, n):
    """Apply the n-point rule on every piece between consecutive edges. Cost: n values per piece.

    Put an edge exactly where the function stops being smooth."""
    if len(edges) < 2 or any(b <= a for a, b in zip(edges[:-1], edges[1:])):
        raise ValueError("edges must be increasing, with at least two entries")
    return float(sum(gauss_legendre(f, a, b, n) for a, b in zip(edges[:-1], edges[1:])))

## Step 3: why 2n - 1?
An n-point rule has 2n numbers to choose (n points, n weights). A polynomial of degree 2n - 1
has 2n coefficients, so we can ask for exactness on all of them: that is what the Legendre
roots achieve. Equally spaced points with chosen weights reach only about degree n - 1 (n
for Simpson's three points). `monomial_errors` tests the rule on x^k over [0, 1] (exact area
1/(k + 1)): the error is at round-off up to degree 2n - 1 and then jumps.

In [ ]:
def monomial_errors(n, max_degree=20):
    """Absolute error of the n-point rule for the area of x^k on [0, 1], k = 0 ... max_degree."""
    return [abs(gauss_legendre(lambda x, k=k: x ** k, 0.0, 1.0, n) - 1.0 / (k + 1)) for k in range(max_degree + 1)]


def exact_degree(n, tol=1e-14, max_degree=40):
    """Highest degree k such that the rule is exact (error below tol) for x^j, every j <= k."""
    k = -1
    for j, e in enumerate(monomial_errors(n, max_degree)):
        if e > tol:
            break
        k = j
    return k

## Step 4: four functions with known areas
The same four as in Session 4, so the rules can be compared with Romberg and Simpson:
e^x on [0, 1], the bell curve exp(-x^2) on [0, 2], sqrt(x) on [0, 1] (not smooth at 0) and
the periodic exp(sin(2 pi x)) over one period. Number 5 is the corner |x - 0.37| on [0, 1].

In [ ]:
def bessel_i0(x):
    """Modified Bessel function I0 by its power series: the exact area of exp(sin(2 pi t)) over a period is I0(1)."""
    term, total, k = 1.0, 1.0, 0
    while term > 1e-18 * total:
        k += 1
        term *= (x / 2.0) ** 2 / (k * k)
        total += term
    return total


KINK = 0.37
FUNCTIONS = {
    1: ("e^x on [0, 1]", math.exp, 0.0, 1.0, math.e - 1.0),
    2: ("exp(-x^2) on [0, 2]", lambda x: math.exp(-x * x), 0.0, 2.0, math.sqrt(math.pi) / 2.0 * math.erf(2.0)),
    3: ("sqrt(x) on [0, 1]", math.sqrt, 0.0, 1.0, 2.0 / 3.0),
    4: ("exp(sin(2 pi x)) on [0, 1]", lambda x: math.exp(math.sin(2.0 * math.pi * x)), 0.0, 1.0, bessel_i0(1.0)),
    5: ("|x - 0.37| on [0, 1]", lambda x: abs(x - KINK), 0.0, 1.0, (KINK ** 2 + (1.0 - KINK) ** 2) / 2.0),
}


def gauss_error(number, n):
    """Absolute error of the n-point rule for function `number`."""
    _, f, a, b, exact = FUNCTIONS[number]
    return abs(gauss_legendre(f, a, b, n) - exact)

## Step 5: the evaluation budget (used by the mini-project)
How many values does Gauss need for a target error? And if you do not know in advance which
n to use, you can raise n until two successive rules agree. But the rules are **not nested**:
the 6-point rule shares no points with the 5-point rule, so every rule you try costs n new
values. `gauss_tol` counts the whole bill.

In [ ]:
def gauss_values_needed(number, target, max_n=40):
    """Smallest n whose error is at most `target` (needs the exact answer), or None."""
    for n in range(1, max_n + 1):
        if gauss_error(number, n) <= target:
            return n
    return None


def gauss_tol(f, a, b, tol=1e-10, n_start=2, max_n=60):
    """Raise n until the n-point and (n+1)-point rules differ by less than tol.

    Returns (value of the (n+1)-point rule, n + 1, total values computed on the way, converged)."""
    spent = 0
    previous = gauss_legendre(f, a, b, n_start)
    spent += n_start
    for n in range(n_start + 1, max_n + 1):
        current = gauss_legendre(f, a, b, n)
        spent += n
        if abs(current - previous) < tol:
            return current, n, spent, True
        previous = current
    return previous, max_n, spent, False


def observed_order(hs, errors):
    """Slope of log(error) against log(h), fitted by least squares (use h = 1/n for algebraic convergence)."""
    return float(np.polyfit(np.log(hs), np.log(errors), 1)[0])

## See it
Figure 1: the 5 Gauss points and weights on [0, 1], drawn as weighted rectangles, next to
5 equally spaced Simpson points. Figure 2: error against function values for Simpson,
Romberg (from Session 4) and Gauss on e^x and on the bell curve. Figure 3: error of the
n-point rule for x^k, jumping after degree 2n - 1; and where Gauss fails.

In [ ]:
def simpson_values(f, a, b, m):
    """Simpson's rule on m values (m odd, m >= 3)."""
    n = m - 1
    h = (b - a) / n
    s = f(a) + f(b) + sum((4 if i % 2 else 2) * f(a + i * h) for i in range(1, n))
    return h / 3.0 * s


def trapezoid_values(f, a, b, m):
    """Trapezoid rule on m equally spaced values (m >= 2)."""
    n = m - 1
    h = (b - a) / n
    return h * (0.5 * (f(a) + f(b)) + sum(f(a + i * h) for i in range(1, n)))


def romberg_corner(f, a, b, levels):
    """Corner of the Romberg table with `levels` rows (2^(levels-1) + 1 values), as in Session 4."""
    h = b - a
    total = 0.5 * (f(a) + f(b))
    rows = [[h * total]]
    for k in range(1, levels):
        h /= 2.0
        total += sum(f(a + (2 * j - 1) * h) for j in range(1, 2 ** (k - 1) + 1))
        row = [h * total]
        for j in range(1, k + 1):
            row.append((4 ** j * row[j - 1] - rows[k - 1][j - 1]) / (4 ** j - 1))
        rows.append(row)
    return rows[-1][-1]


if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    BLUE, ORANGE, MAGENTA = "#0072b2", "#c86a00", "#d81b60"
    t5, w5 = gauss_legendre_nodes(5)
    print("5-point nodes on [0, 1]:", np.round(0.5 + 0.5 * t5, 5), "weights", np.round(0.5 * w5, 5))
    print(f"e^x on [0, 1]: 5-point Gauss error {gauss_error(1, 5):.1e}; Simpson with 5 values {abs(simpson_values(math.exp, 0, 1, 5) - (math.e - 1)):.1e}")
    for n in range(1, 8):
        print(f"  n = {n}: error {gauss_error(1, n):.1e}, exact up to degree {exact_degree(n)}")
    for target in (1e-6, 1e-8, 1e-10, 1e-12):
        _, f2, a2, b2, _ = FUNCTIONS[2]
        v, nn, spent, ok = gauss_tol(f2, a2, b2, target)
        print(f"bell curve, target {target:.0e}: Gauss needs n = {gauss_values_needed(2, target)}; with the stopping rule it stops at n = {nn} having computed {spent} values")
    ns_s = [20, 40, 80, 160]
    print("sqrt(x): errors", [f"{gauss_error(3, n):.1e}" for n in ns_s], "observed order", round(observed_order([1.0 / n for n in ns_s], [gauss_error(3, n) for n in ns_s]), 2))
    print("periodic: Gauss errors n = 5, 8, 10:", [f"{gauss_error(4, n):.1e}" for n in (5, 8, 10)])
    print("corner: one rule on [0, 1] with 10 points", f"{gauss_error(5, 10):.1e}", "; split at 0.37 with 1 point per piece:",
          f"{abs(composite_gauss(FUNCTIONS[5][1], [0.0, KINK, 1.0], 1) - FUNCTIONS[5][4]):.1e}")

    # figure 1: nodes and weights
    fig, axs = plt.subplots(1, 2, figsize=(10, 4.0), sharey=True)
    xx = np.linspace(0, 1, 400)
    xg, wg = 0.5 + 0.5 * t5, 0.5 * w5
    xs = np.linspace(0, 1, 5)
    for ax, xp, label in ((axs[0], xs, "5 equally spaced values (Simpson)"), (axs[1], xg, "5 Gauss-Legendre values")):
        ax.plot(xx, np.exp(xx), color=BLUE, lw=2)
        ax.plot(xp, np.exp(xp), "o", color=MAGENTA, ms=7)
        ax.set_title(label, fontsize=10)
        ax.set_xlabel("x")
        ax.grid(alpha=0.3)
    for xi, wi in zip(xg, wg):
        axs[1].bar(xi, np.exp(xi), width=wi, color=ORANGE, alpha=0.35, edgecolor=ORANGE)
    axs[0].set_ylabel("f(x) = e^x")
    axs[0].text(0.03, 0.2, f"error {abs(simpson_values(math.exp, 0, 1, 5) - (math.e - 1)):.1e}", transform=axs[0].transAxes)
    axs[1].text(0.03, 0.2, f"error {gauss_error(1, 5):.1e}", transform=axs[1].transAxes)
    fig.tight_layout()
    fig.savefig(OUT / "nodes_weights.png", dpi=120)

    # figure 2: error against values
    fig2, axs2 = plt.subplots(1, 2, figsize=(10, 4.0))
    for ax, number in zip(axs2, (1, 2)):
        name, f, a, b, exact = FUNCTIONS[number]
        gs = list(range(1, 17))
        ax.semilogy(gs, [max(gauss_error(number, n), 1e-17) for n in gs], "^-", color=BLUE, label="Gauss-Legendre")
        ms = [3, 5, 9, 17, 33]
        ax.semilogy(ms, [max(abs(simpson_values(f, a, b, m) - exact), 1e-17) for m in ms], "s-", color=ORANGE, label="Simpson")
        ax.semilogy(ms, [max(abs(romberg_corner(f, a, b, int(math.log2(m - 1)) + 1) - exact), 1e-17) for m in ms], "o-", color=MAGENTA, label="Romberg")
        ax.axhline(2.2e-16, color="#666666", ls=":", lw=1)
        ax.set_xlabel("function values")
        ax.set_ylabel("absolute error")
        ax.set_title(name, fontsize=10)
        ax.set_ylim(1e-17, 1)
        ax.grid(alpha=0.3, which="both")
        ax.legend(fontsize=8, loc="lower left")
    fig2.tight_layout()
    fig2.savefig(OUT / "error_vs_values.png", dpi=120)

    # figure 3: exactness by degree, and the failures
    fig3, axs3 = plt.subplots(1, 3, figsize=(13, 3.9))
    for n, col, mk in ((2, MAGENTA, "o"), (3, ORANGE, "s"), (5, BLUE, "^")):
        axs3[0].semilogy(range(21), [max(e, 1e-17) for e in monomial_errors(n)], mk + "-", color=col, ms=4, label=f"n = {n} (exact to degree {2 * n - 1})")
    axs3[0].set_xlabel("degree k of x^k")
    axs3[0].set_ylabel("absolute error of the area")
    axs3[0].set_title("Exactness: error jumps after degree 2n - 1", fontsize=10)
    axs3[0].legend(fontsize=8, loc="lower right")
    ns = np.arange(2, 81)
    axs3[1].loglog(ns, [gauss_error(3, int(n)) for n in ns], "^-", ms=3, color=BLUE, label="Gauss-Legendre")
    axs3[1].loglog([5, 9, 17, 33, 65], [abs(simpson_values(math.sqrt, 0, 1, m) - 2 / 3) for m in (5, 9, 17, 33, 65)], "s-", color=ORANGE, label="Simpson")
    axs3[1].set_xlabel("function values")
    axs3[1].set_xticks([2, 5, 10, 20, 40, 80], ["2", "5", "10", "20", "40", "80"])
    axs3[1].minorticks_off()
    axs3[1].set_title("sqrt(x): only algebraic decay (slope -3)", fontsize=10)
    axs3[1].legend(fontsize=8)
    f4, ex4 = FUNCTIONS[4][1], FUNCTIONS[4][4]
    axs3[2].semilogy(range(2, 21), [max(gauss_error(4, n), 1e-17) for n in range(2, 21)], "^-", ms=3, color=BLUE, label="Gauss-Legendre")
    ev = [2, 3, 5, 9, 17]
    axs3[2].semilogy(ev, [max(abs(trapezoid_values(f4, 0.0, 1.0, m) - ex4), 1e-17) for m in ev], "o-", color=MAGENTA, label="trapezoid")
    axs3[2].set_xlabel("function values")
    axs3[2].set_title("periodic: the trapezoid rule wins", fontsize=10)
    axs3[2].legend(fontsize=8, loc="lower left")
    for ax in axs3:
        ax.grid(alpha=0.3, which="both")
        ax.set_ylim(1e-17, 1)
    fig3.tight_layout()
    fig3.savefig(OUT / "exactness_and_failures.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint. The mini-project (README) asks how many function values each rule needs.

## Coding tasks
1. **Easy.** Call `gauss_legendre_nodes(4)` and check by hand that the nodes are symmetric, that the weights add up to 2, and that the 4-point rule gives the exact area of x^7 over [-1, 1] (zero) but not of x^8 (exact 2/9).
2. **Stretch.** For e^x on [0, 1] use `gauss_values_needed` to find the number of Gauss values for the targets 1e-6, 1e-8, 1e-10 and 1e-12, and compare with the Romberg counts from Session 4 (`evaluations_needed`). Which rule is cheaper at each target?
3. **Explain.** Explain why the 6-point Gauss rule cannot reuse the 5 values of the 5-point rule, why that matters when you do not know in advance how many points you need, and what a composite rule (`composite_gauss`) with an edge at a corner does about a function that is not smooth.

## Assessment
- **A. Conceptual.** Why is the n-point Gauss-Legendre rule exact up to degree 2n - 1, when the n-point rule with equally spaced points is exact only up to about degree n - 1?
- **B. Numerical.** Use the 2-point Gauss-Legendre rule to find the area of x^3 + 1 on [1, 3], and then the area of x^4 on [1, 3]. The exact answers are 22 and 48.4.
- **C. Graph and simulation interpretation.** In the error-against-values figure for the bell curve, at 9 values the errors are 1.6e-5 (Simpson), 5.0e-5 (Romberg) and 1.2e-11 (Gauss). What do you conclude about Gauss, and about the claim that Romberg is always better than Simpson?
- **D. Design and optimisation challenge.** You can afford 10 function values (10 minutes) for the area of exp(-x^2) on [0, 2]. Which rule do you choose and what error do you expect? Then suppose you needed 1e-12 and did not know how many points to use: what would the Gauss approach cost, and how does it compare with Romberg?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math

import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    ex = math.e - 1.0
    # hand calculations: Legendre polynomials and the small rules
    p2, dp2 = L.legendre_and_derivative(2, 0.5)
    checks.append((f"Legendre P2(0.5) = (3 x 0.25 - 1) / 2 = {p2:.3f} and P2'(0.5) = 3 x 0.5 = {dp2:.2f}", abs(p2 + 0.125) < 1e-12 and abs(dp2 - 1.5) < 1e-12))
    t2, w2 = L.gauss_legendre_nodes(2)
    t3, w3 = L.gauss_legendre_nodes(3)
    checks.append(("2-point rule: nodes +-1/sqrt(3) = +-0.577350, weights 1 and 1",
                   np.allclose(t2, [-1 / math.sqrt(3), 1 / math.sqrt(3)], atol=1e-14) and np.allclose(w2, [1, 1], atol=1e-14)))
    checks.append(("3-point rule: nodes -sqrt(0.6), 0, +sqrt(0.6) = -0.774597, 0, 0.774597; weights 5/9, 8/9, 5/9",
                   np.allclose(t3, [-math.sqrt(0.6), 0, math.sqrt(0.6)], atol=1e-14) and np.allclose(w3, [5 / 9, 8 / 9, 5 / 9], atol=1e-14)))
    g2, g3 = L.gauss_legendre(math.exp, 0.0, 1.0, 2), L.gauss_legendre(math.exp, 0.0, 1.0, 3)
    checks.append((f"worked example: 2 points give {g2:.6f} (error {g2 - ex:.1e}), 3 points give {g3:.6f} (error {g3 - ex:.1e})",
                   abs(g2 - 1.717896) < 1e-6 and abs(g3 - 1.718281) < 1e-6 and abs(abs(g2 - ex) - 3.85e-4) < 1e-5 and abs(abs(g3 - ex) - 8.24e-7) < 1e-8))
    # nodes and weights against numpy
    worst, ok = 0.0, True
    for n in range(1, 21):
        t, w = L.gauss_legendre_nodes(n)
        tn, wn = np.polynomial.legendre.leggauss(n)
        worst = max(worst, float(np.max(np.abs(t - tn))), float(np.max(np.abs(w - wn))))
        ok &= bool(np.all(np.diff(t) > 0) and np.all(np.abs(t) < 1) and np.all(w > 0) and abs(w.sum() - 2.0) < 1e-13 and np.allclose(t, -t[::-1], atol=1e-14))
    checks.append((f"nodes and weights for n = 1 ... 20: sorted, inside (-1, 1), positive weights summing to 2, symmetric, and equal to numpy's to {worst:.0e}", ok and worst < 1e-13))
    # degree of exactness
    degs = [L.exact_degree(n) for n in range(1, 9)]
    checks.append((f"the n-point rule is exact up to degree 2n - 1 (n = 1 ... 8 gives {degs}) and not beyond",
                   degs == [1, 3, 5, 7, 9, 11, 13, 15] and L.monomial_errors(3)[6] > 1e-4))
    # interval mapping
    checks.append(("any interval: x^2 on [2, 5] is 39 with 2 points", abs(L.gauss_legendre(lambda x: x * x, 2.0, 5.0, 2) - 39.0) < 1e-12))
    # counting evaluations
    calls = [0]

    def counted(x):
        calls[0] += 1
        return math.exp(x)

    L.gauss_legendre(counted, 0.0, 1.0, 5)
    checks.append((f"the 5-point rule costs {calls[0]} function values", calls[0] == 5))
    # the roadmap checkpoint
    simp5 = abs((0.25 / 3.0) * (1 + 4 * math.exp(0.25) + 2 * math.exp(0.5) + 4 * math.exp(0.75) + math.e) - ex)
    err5 = abs(L.gauss_legendre(math.exp, 0.0, 1.0, 5) - ex)
    checks.append((f"roadmap checkpoint: e^x on [0, 1] with 5 function values has error {err5:.1e} (below 1e-11); Simpson with 5 values gives {simp5:.1e}",
                   err5 < 1e-11 and 3.6e-5 < simp5 < 3.8e-5))
    err4 = abs(L.gauss_legendre(math.exp, 0.0, 1.0, 4) - ex)
    checks.append((f"4 values are not enough for the roadmap target (error {err4:.1e}, above 1e-11)", err4 > 1e-11))
    # the bell curve budget
    need = [L.gauss_values_needed(2, t) for t in (1e-6, 1e-8, 1e-10, 1e-12)]
    checks.append((f"mini-project: the bell curve needs {need} Gauss values for 1e-6, 1e-8, 1e-10, 1e-12 (Romberg needs 17, 33, 33, 65; Simpson 17, 65, 257, 513)",
                   need == [6, 7, 9, 10]))
    f2, a2, b2 = L.FUNCTIONS[2][1], L.FUNCTIONS[2][2], L.FUNCTIONS[2][3]
    stop = [L.gauss_tol(f2, a2, b2, t) for t in (1e-6, 1e-12)]
    checks.append((f"not nested: the stopping rule at 1e-6 ends at n = {stop[0][1]} after {stop[0][2]} values, and at 1e-12 at n = {stop[1][1]} after {stop[1][2]} values (the same 65 as Romberg)",
                   stop[0][1:] == (7, 27, True) and stop[1][1:] == (11, 65, True)))
    # smooth enough, fast: e^x
    checks.append(("round-off floor: 7 points on e^x are at 2e-16", L.gauss_error(1, 7) < 1e-15))
    # failure: sqrt(x)
    ns = [20, 40, 80, 160]
    ord_sqrt = L.observed_order([1.0 / n for n in ns], [L.gauss_error(3, n) for n in ns])
    checks.append((f"failure: sqrt(x) falls only like n^-3 (observed order {ord_sqrt:.2f}); 160 values give {L.gauss_error(3, 160):.1e}, not round-off",
                   2.8 < ord_sqrt < 3.2 and L.gauss_error(3, 160) > 1e-9))
    # failure: periodic
    f4, ex4 = L.FUNCTIONS[4][1], L.FUNCTIONS[4][4]
    trap17 = abs(L.trapezoid_values(f4, 0.0, 1.0, 17) - ex4)
    checks.append((f"failure: periodic function: the trapezoid rule with 17 values is at round-off ({trap17:.0e}), Gauss with 17 values has error {L.gauss_error(4, 17):.1e}",
                   trap17 < 1e-14 and L.gauss_error(4, 17) > 1e-11))
    # failure: corner and the fix
    kink_f, kink_exact = L.FUNCTIONS[5][1], L.FUNCTIONS[5][4]
    one = abs(L.gauss_legendre(kink_f, 0.0, 1.0, 10) - kink_exact)
    split = abs(L.composite_gauss(kink_f, [0.0, L.KINK, 1.0], 1) - kink_exact)
    checks.append((f"failure and fix: |x - 0.37| with one 10-point rule has error {one:.1e}; split at 0.37 with 1 point per piece (2 values) the error is {split:.0e}",
                   one > 1e-3 and split < 1e-14))
    # failure: narrow peak
    peak = lambda x: math.exp(-((x - 0.35) / 0.005) ** 2)
    peak_exact = 0.005 * math.sqrt(math.pi) / 2.0 * (math.erf((1 - 0.35) / 0.005) - math.erf((0 - 0.35) / 0.005))
    miss = abs(L.gauss_legendre(peak, 0.0, 1.0, 20) - peak_exact) / peak_exact
    checks.append((f"failure: a narrow peak (width 0.005) is missed by 20 Gauss points (error {100 * miss:.0f} percent)", miss > 0.99))
    # exact areas
    checks.append(("exact areas: sqrt(pi)/2 erf(2) = 0.882081 and I0(1) = 1.266066", abs(L.FUNCTIONS[2][4] - 0.882081390762) < 1e-10 and abs(L.FUNCTIONS[4][4] - 1.2660658777520082) < 1e-14))
    # limits
    bad = 0
    for call in (lambda: L.gauss_legendre_nodes(0), lambda: L.gauss_legendre(math.exp, 1.0, 1.0, 3),
                 lambda: L.composite_gauss(math.exp, [0.0], 3), lambda: L.composite_gauss(math.exp, [0.0, 1.0, 0.5], 3)):
        try:
            call()
        except ValueError:
            bad += 1
    checks.append(("limits: zero points, a >= b, and edges that are not increasing are rejected", bad == 4))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
